# Building model

Now that we have performed exploratory data analysis (EDA) in a [separate notebook](./understanding_data.ipynb), we can move on to building and testing models.

## Step 0: Basic prep

Get the main input directory.

In [1]:
from pathlib import Path

data_dir = Path('../input/jane-street-real-time-market-data-forecasting')

## Step 1: Data details

- From our preliminary EDA, we found that that 3 features, namely, `feature_09`, `feature_10`, and `feature_11`, are categorical features, with categories represented as numbers, seemingly $\leq$ 20.
    - Note, however, that these were results from the `partition_id=0` training data; we should investigate this for all partitions to make sure we get the correct number of categories per feature
- Similarly, we need to be able to adequately encode the `symbol_id` values, so we have to go through the whole data to make sure we know how many there are
    - Let's not forget to make sure our data processing steps account for the possibility of a new `symbol_id` value, not originally present in the training data. 
- We also observed a number of features filled completely with `NaN` values; these features will be removed. For the other features, imputation with zero (0.0) will be used

### Step 1.0: Scan the data

In [2]:
import polars as pl

training_scan = pl.scan_parquet(data_dir / 'train.parquet')

### Step 1.1: Get unique values

Now, let's get the unique values for each entry of interest (including `NaN`)

In [3]:
# Specify the categorical fields
categorical_fields = [f'feature_{i:02d}' for i in [9, 10, 11]] + ['symbol_id']

# Initialize a dictionary to keep the categories
categorical_fields = dict.fromkeys(categorical_fields)

for field in categorical_fields.keys():
    # Get the info
    categories = set(training_scan.select(pl.col(field).unique()).collect()[field].to_list())
    categorical_fields[field] = categories
    print(f"Field {field} has {len(categories)} distinct categories:")
    print(categories)
    print('=' * 120)

Field feature_09 has 22 distinct categories:
{2, 4, 9, 11, 12, 14, 15, 25, 26, 30, 34, 42, 44, 46, 49, 50, 57, 64, 68, 70, 81, 82}
Field feature_10 has 9 distinct categories:
{1, 2, 3, 4, 5, 6, 7, 10, 12}
Field feature_11 has 31 distinct categories:
{388, 261, 9, 522, 11, 13, 16, 150, 534, 24, 25, 410, 539, 158, 159, 34, 40, 297, 171, 48, 50, 59, 62, 63, 66, 195, 76, 336, 214, 230, 376}
Field symbol_id has 39 distinct categories:
{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38}


### Step 1.2: Recompute `NaN` feature for the entire dataset

Recall several features are fully `NaN` in partition 0; we should ignore those.

Those were:
- feature_00
- feature_21
- feature_02
- feature_03
- feature_04
- feature_01
- feature_31
- feature_27
- feature_26

Let's make sure this trend remains for all partitions first. We will want to make sure that, for the features we keep that have more than 5% `NULL` entries, we create an additional "missing" flag to indicate to the model the value was missing.

In [4]:
# Get the total number of entries
total_num_entries = training_scan.select(pl.len()).collect().item()

# Get the counts
training_null_counts = training_scan.null_count().collect()

# Prepare dictionary to keep the features with more than 5% NULL entries
features_with_non_negligible_null = {}
large_null_fraction_threshold = 0.05

for feature in [f'feature_{i:02d}' for i in range(79)]:
    fraction_null = training_null_counts[feature].item() / total_num_entries
    if fraction_null >= large_null_fraction_threshold:
        print(f'{feature} has {100*fraction_null:.1f}% NULL entries')
        features_with_non_negligible_null[feature] = fraction_null

feature_00 has 6.8% NULL entries
feature_01 has 6.8% NULL entries
feature_02 has 6.8% NULL entries
feature_03 has 6.8% NULL entries
feature_04 has 6.8% NULL entries
feature_21 has 17.9% NULL entries
feature_26 has 17.9% NULL entries
feature_27 has 17.9% NULL entries
feature_31 has 17.9% NULL entries
feature_39 has 9.1% NULL entries
feature_42 has 9.1% NULL entries
feature_50 has 9.0% NULL entries
feature_53 has 9.0% NULL entries


Clearly, our EDA on partition 0 alone was not enough: as we see, the features that were previously fully absent are now present! Therefore, we should keep all the features, but treat these ones carefully.

## Step 2: Create data processing steps

Now that we have better information about critical parts of the data for the whole training set, let's create the necesary steps to processes this data to serve as model inputs.

### Step 2.0: Model details

Seeing as the goal of the data processor is to serve data to the model, we first need to delineate how the model will work. 

Since I am using this as a way to learn, I want to use a recurrent neural network (RNN)-based strategy. It is not something I have done in the past, but it somewhat resembles some of my previous work in Kálmán and particl filtering, and hidden Markov models (HMM). However, one of the primary difficulties with using an out-of-the-box RNN is that, at future prediction times, we may be given some non-zero amount of symbols that were not present during training (and may not be provided features for the symbols we already saw).

To circumvent this issue, I decided to take some inspiration from online forums, and the general idea will be the same: having a hidden state that is used for predictions, and updated from timestep to timestep. However, at a given timestep, the update will be a bit more involved, following these steps:
1. Gather all of the (79) features for all of the known (39) symbols
2. Create "market"-like symbols, which track, feature-by-feature, the daily and timestamp average, standard deviation, minimum, maximum, and count, over all symbols (known and unknown). That means, ten (10) "new" symbols are created, each also containing 79 features (recall some of these will have to be one-hot encoded, so the dimensionality will be larger than 79).
    - NOTE: for these operations, missing values should be ignored, rather than imputed.
    - NOTE: the new "market"-stats symbols would be `daily_mean_so_far`, `daily_std_so_far`, `daily_min_so_far`, `daily_max_so_far`, `log1p(daily_count_so_far)`, `mean_now`, `std_now`, `min_now`, `max_now`, `log1p(count_now)`
    - NOTE: other than the mean, the other statistics are pretty meaningless for the one-hot encodded categories. However, we will keep them to not overly complicated this process.
3. Impute missing features with 0.
4. Prepare the first input to the model, containing 49 symbols (39 known + 10 market statistics) with 79 zero-imputed features each (again, recall the categorical features will increase the actual dimensionality along the feature dimension). Let's call this matrix $M_t$, since it somewhat corresponds to the market at time $t$
    - NOTE: critically, $M_t$ is missing exact information of the unknown symbols! However, with the statistics of the current timestamp, this is partially mitigated.
5. $M_t$ is used as input to an attention-head, which is used to encode this information in some latent representation $L_t$
6. The latent embedding $L_t$ is combined with the present timestamp $T_t$, as well as with the previous timestamp $T_{t-1}$, and used to update the hidden state $h_{t-1}$ to $h_t$ through some form of gated recurrent unit (GRU) architecture
7. Each symbol provided is then passed through its own embedding, combined with its 79 features to form another latent represenation of this symbol, which is then combined with the updated hidden state $h_t$ to produce a prediction. This operation would likely be something like `concat([symbol_embedding, features]) --> MLP --> latent_rep --> concat([latent_rep, h_t]) --> MLP --> prediction`